# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

In [1]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Push Prices Handler loaded at 2026-04-02 17:24:17 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-02


In [2]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module.ipynb
os.chdir('..')

print('Loading market data...')
market_data = get_market_data()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
    GROUP BY ALL

'''
df_prices = query_snowflake(PRICES_QUERY)
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Market Data Module loaded at 2026-04-02 17:24:18 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
Loading market data...

FETCHING

/tmp/ipykernel_23555/3473350795.py:139: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


    Records after group processing: 25849

Step 5: Adding WAC and margin data...
    Records with WAC: 25507

Step 6: Filtering by price coverage...
    Records after price coverage filter: 15467

Step 7: Calculating price percentiles...
    Records after price analysis: 14848

Step 8: Converting prices to margins...

MARKET DATA COMPLETE
Total records: 14848
  - With marketplace prices: 12524
  - With scrapped prices: 8586
  - With Ben Soliman prices: 12021
  Market data: 14848 rows
Loading WAC...
  WAC: 8390 rows
Loading current prices...
  Prices: 112123 rows
Loading packing units...
  Packing units: 35917 rows
Loading product info...
  Products: 25605 rows
Loading target margins...
  Target margins: 481 rows

All data loaded.


In [3]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

price_cols = ['minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum']
market_subset = market_data[['product_id', 'cohort_id'] + [c for c in price_cols if c in market_data.columns]].copy()

# Lookup is per (product_id, cohort_id) since market prices vary by cohort
lookup = df_products.merge(df_wac, on='product_id', how='left')

# Expand products × cohorts, then merge cohort-specific market prices
cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(market_subset, on=['product_id', 'cohort_id'], how='left')
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left')
lookup['target_margin'] = lookup['target_margin'].fillna(0.10)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

if 'minimum' in lookup.columns and 'maximum' in lookup.columns:
    lookup['market_avg'] = (lookup['minimum'] + lookup['maximum']) / 2

print(f'Lookup table: {len(lookup)} rows (product × cohort)')
print(f'  With market data: {lookup["minimum"].notna().sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 230495 rows (product × cohort)
  With market data: 14857
  With WAC: 75560
  With target margin: 230495


In [6]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin', or a number (fixed price)
#
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.

PUSH_LIST = [
(130,'market_50'),(305,'market_50'),(143,'market_50'),(151,'market_50'),(2049,'market_50'),(144,'market_50'),(9507,'market_50'),(141,'market_50'),(145,'market_50'),(146,'market_50'),(205,'market_50'),(362,'market_50'),(7004,'market_50'),(2327,'market_50'),(2778,'market_50'),(7885,'market_50'),(159,'market_50'),(248,'market_50'),(2328,'market_50'),(6935,'market_50'),(8648,'market_50'),(8650,'market_50'),(326,'market_50'),(206,'market_50'),(7630,'market_50'),(997,'market_50'),(75,'market_50'),(3,'market_50'),(998,'market_50'),(990,'market_50'),(7005,'market_50'),(361,'market_50'),(589,'market_50'),(7886,'market_50'),(996,'market_50'),(88,'market_50'),(417,'market_50'),(201,'market_50'),(593,'market_50'),(83,'market_50'),(5493,'market_50'),(1504,'market_50'),(615,'market_50'),(2424,'market_50'),(8915,'market_50'),(12224,'market_50'),(5491,'market_50'),(161,'market_50'),(412,'market_50'),(8672,'market_50'),(2423,'market_50'),(3893,'market_50'),(995,'market_50'),(616,'market_50'),(346,'market_50'),(142,'market_50'),(3889,'market_50'),(6936,'market_50'),(1776,'market_50'),(2912,'market_50'),(439,'market_50'),(6097,'market_50'),(413,'market_50'),(147,'market_50'),(1309,'market_50'),(5151,'market_50'),(1124,'market_50'),(3891,'market_50'),(2054,'market_50'),(5005,'market_50'),(2775,'market_50'),(7887,'market_50'),(8649,'market_50'),(74,'market_50'),(2050,'market_50'),(1007,'market_50'),(10149,'market_50'),(1163,'market_50'),(2991,'market_50'),(61,'market_50'),(972,'market_50'),(7084,'market_50'),(1413,'market_50'),(1654,'market_50'),(1656,'market_50'),(10574,'market_50'),(2055,'market_50'),(8943,'market_50'),(140,'market_50'),(2776,'market_50'),(2057,'market_50'),(345,'market_50'),(11737,'market_50'),(6098,'market_50'),(5208,'market_50'),(9760,'market_50'),(418,'market_50'),(957,'market_50'),(7036,'market_50'),(822,'market_50')

]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'minimum',
    'market_25': 'percentile_25',
    'market_50': 'percentile_50',
    'market_75': 'percentile_75',
    'market_max': 'maximum',
    'market_avg': 'market_avg',
}

results = []
for product_id, action in PUSH_LIST:
    product_rows = lookup[lookup['product_id'] == product_id]
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.10)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
print()
df_results[df_results['status'] == 'OK']

Results: 853 OK across 94 products × cohorts, 56 errors/skipped

Errors/Skipped:
 product_id  cohort_id                                  sku    action         status
        159        703 مرقة دجاج كنور فاين فودز شريط - 8 جم market_50 NO MARKET DATA
       2424        700     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424        701     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424        702     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424        703     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424        704     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424       1123     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424       1124     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424       1125     حفاضات جود كير مقاس 5 - 40 حفاضة market_50 NO MARKET DATA
       2424       1126     حفاضات جود كير مقاس 5 - 40 حفاضة market_50

,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,130,700,لبن بخيره - 500 مل,بخيره,market_50,483.433768,503.25,506.7350,506.75,0.046011,OK
1,130,701,لبن بخيره - 500 مل,بخيره,market_50,483.433768,503.25,506.7350,506.75,0.046011,OK
2,130,702,لبن بخيره - 500 مل,بخيره,market_50,483.433768,498.00,505.8675,505.75,0.044125,OK
3,130,703,لبن بخيره - 500 مل,بخيره,market_50,483.433768,502.50,506.7350,506.75,0.046011,OK
4,130,704,لبن بخيره - 500 مل,بخيره,market_50,483.433768,500.75,505.8675,505.75,0.044125,OK
...,...,...,...,...,...,...,...,...,...,...,...
886,957,704,لبن المراعى تريتس شيكولاتة - 200 مل,المراعي البان,market_50,270.256385,275.00,283.0000,283.00,0.045030,OK
887,957,1123,لبن المراعى تريتس شيكولاتة - 200 مل,المراعي البان,market_50,270.256385,275.00,283.0000,283.00,0.045030,OK
888,957,1124,لبن المراعى تريتس شيكولاتة - 200 مل,المراعي البان,market_50,270.256385,275.00,283.0000,283.00,0.045030,OK
889,957,1125,لبن المراعى تريتس شيكولاتة - 200 مل,المراعي البان,market_50,270.256385,275.00,283.0000,283.00,0.045030,OK


In [17]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()
if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 94 products × 9 cohorts = 1137 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 1137
Price changes to push: 1120
Skipped (no change): 17

Price changes summary:
  Increases: 1112
  Decreases: 8

🔗 Mirrored prices to 6 main/general cohorts (+719 rows)
   Cohort 695 ← 700: 120 rows
   Cohort 61 ← 700: 120 rows
   Cohort 699 ← 702: 121 rows
   Cohort 697 ← 703: 119 rows
   Cohort 698 ← 704: 121 rows
   Cohort 696 ← 1123: 118 rows

📋 Prepared 1799 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (120 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 78.48it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (120 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 80.01it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (120 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 85.56it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 1799
Total failed: 0
  Cleanup: removed 30 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 1137, 'price_changes': 1120, 'pushed': 1799, 'failed': 0, 'skipped': 17, 'source_module': 'manual_push', 'timestamp': '2026-04-02 17:30:42', 'mode': 'live', 'skipped_cohorts': []}
